In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE-CELL PIPELINE: Feature Extraction + Dataset Build (SHARMA 2021 METHOD)
# Paste this entire block into ONE Kaggle notebook cell and run.
# ═══════════════════════════════════════════════════════════════════════════════

# ── 0. DEPENDENCIES ────────────────────────────────────────────────────────────
import subprocess, sys
# Replaced antropy with PyWavelets
subprocess.check_call([sys.executable, "-m", "pip", "install", "PyWavelets", "-q"])

# ── 1. IMPORTS ─────────────────────────────────────────────────────────────────
from __future__ import annotations
import math
import json
import warnings
import numpy as np
import pywt
from pathlib import Path
from collections import defaultdict
from typing import Tuple

# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
SFREQ: float      = 256.0
EPOCH_SEC: float  = 30.0
N_SAMPLES: int    = int(SFREQ * EPOCH_SEC)   # 7680

# 17-channel layout from signal_processing.py:
# idx: 0=F3  1=F4  2=C3  3=C4  4=O1  5=O2  6=A1  7=A2
#      8=C4A1 9=C3A2 10=F3A2 11=F4A1 12=O1A2 13=O2A1
#      14=EMG 15=EMG1 16=EMG2
EEG_DIFF_INDICES: list[int] = [8, 9, 10, 11, 12, 13]   # differential only

STAGE_MAP: dict[str, int] = {
    "Wake": 0, "Stage_1": 1, "Stage_2": 2, "Stage_3": 3, "REM": 4
}
GROUP_MAP: dict[str, int] = {
    "Normal": 0, "Insomnia": 1, "insomaniac": 1
}

# ── CONFIGURE YOUR PATHS HERE ──────────────────────────────────────────────────
DATA_DIR   = "/kaggle/input/datasets/johanliebert28/insomnia-sliced-dataset/Ensominia_Processed_Tensors"
OUTPUT_DIR = "/kaggle/working/ml_ready_sharma"

# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — FEATURE EXTRACTION (SHARMA ET AL. 2021)
# ══════════════════════════════════════════════════════════════════════════════

def _check_epoch(epoch: np.ndarray) -> None:
    if epoch.ndim != 2:
        raise ValueError(f"Expected 2D (channels, samples), got {epoch.shape}")
    if epoch.shape[1] != N_SAMPLES:
        raise ValueError(f"Expected {N_SAMPLES} samples, got {epoch.shape[1]}. "
                         f"Check SFREQ={SFREQ}Hz constant.")
    if np.any(np.isnan(epoch)):
        raise ValueError("NaN in epoch — artifact masking failed upstream.")
    if np.any(np.isinf(epoch)):
        raise ValueError("Inf in epoch.")

def compute_sharma_wavelet_norms(epoch: np.ndarray, wavelet: str = 'bior3.3') -> np.ndarray:
    """
    Extracts Sharma 2021 wavelet norm features from an EEG epoch.
    Performs 5-level 1-D wavelet decomposition yielding 6 sub-bands.
    Extracts L1, L2, and L-infinity norms from each sub-band.
    
    I/O: (17, 7680) float32 → (108,) float32
    """
    _check_epoch(epoch)
    eeg = epoch[EEG_DIFF_INDICES, :]  # (6, 7680)
    n_channels = eeg.shape[0]
    
    features = []
    
    for ch in range(n_channels):
        signal = eeg[ch, :]
        
        # 5-level 1D discrete wavelet transform
        # Returns 6 sub-bands: [cA5, cD5, cD4, cD3, cD2, cD1] (Approx + 5 Detailed)
        coeffs = pywt.wavedec(signal, wavelet, level=5)
        
        for sub_band in coeffs:
            # L1 Norm (Sum of absolute values)
            l1_norm = np.sum(np.abs(sub_band))
            
            # L2 Norm (RMS equivalent)
            l2_norm = np.linalg.norm(sub_band, ord=2)
            
            # L-infinity Norm (Peak absolute value)
            linf_norm = np.max(np.abs(sub_band))
            
            features.extend([l1_norm, l2_norm, linf_norm])
            
    return np.array(features, dtype=np.float32)

# ── FEATURE NAME REGISTRY ──────────────────────────────────────────────────────
SUB_BANDS = ["A5", "D5", "D4", "D3", "D2", "D1"]
NORMS = ["L1", "L2", "Linf"]

FEATURE_NAMES: list[str] = [
    f"wvl_{sb}_{norm}_ch{c}" 
    for c in range(6) 
    for sb in SUB_BANDS 
    for norm in NORMS
]

assert len(FEATURE_NAMES) == 108, f"FATAL: Feature count = {len(FEATURE_NAMES)}, expected 108"

# ── MASTER EXTRACTOR ──────────────────────────────────────────────────────────
def extract_features(epoch: np.ndarray) -> np.ndarray:
    """
    I/O: (17, 7680) float32 → (108,) float32
    Raises ValueError on NaN, Inf, or wrong shape.
    """
    return compute_sharma_wavelet_norms(epoch)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION C — FILENAME PARSER
# ══════════════════════════════════════════════════════════════════════════════
def parse_filename(fname: str) -> tuple | None:
    stem  = Path(fname).stem
    parts = stem.split('_')

    if parts[0] == 'insomaniac' and len(parts) >= 4 and parts[1] == 'Subject':
        group = 'insomaniac'
        try:    subj_num = int(parts[2])
        except: return None
        stage = '_'.join(parts[3:])

    elif parts[0] in ('Normal', 'Insomnia') and len(parts) >= 3:
        group = parts[0]
        try:    subj_num = int(parts[1])
        except: return None
        stage = '_'.join(parts[2:])

    else:
        return None

    if stage not in STAGE_MAP:
        print(f"  [SKIP] Unknown stage '{stage}' in {fname}")
        return None

    return group, subj_num, stage

# ══════════════════════════════════════════════════════════════════════════════
# SECTION D — PER-FILE LOADER + EXTRACTOR
# ══════════════════════════════════════════════════════════════════════════════
def load_and_extract(
    npy_path: str, group: str, subj_num: int, stage: str, subj_idx: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    
    tensor = np.load(npy_path)

    if tensor.ndim != 3 or tensor.shape[1] != 17 or tensor.shape[2] != N_SAMPLES:
        raise ValueError(
            f"Shape mismatch: {Path(npy_path).name} → expected (N, 17, {N_SAMPLES}), "
            f"got {tensor.shape}.  If sfreq ≠ 256 Hz update N_SAMPLES."
        )

    features, bad = [], 0
    for i in range(tensor.shape[0]):
        try:
            fv = extract_features(tensor[i])
            if np.any(np.isnan(fv)) or np.any(np.isinf(fv)):
                bad += 1
            else:
                features.append(fv)
        except Exception:
            bad += 1

    if bad:
        print(f"    [WARN] {bad}/{tensor.shape[0]} epochs rejected in {Path(npy_path).name}")

    if not features:
        e = np.empty((0, 108), np.float32)
        return e, np.empty(0, np.int32), np.empty(0, np.int32), np.empty(0, np.int32)

    X   = np.stack(features)
    y_s = np.full(len(features), STAGE_MAP[stage],    dtype=np.int32)
    y_g = np.full(len(features), GROUP_MAP[group],    dtype=np.int32)
    s_i = np.full(len(features), subj_idx,            dtype=np.int32)
    return X, y_s, y_g, s_i

# ══════════════════════════════════════════════════════════════════════════════
# SECTION E — MAIN BUILD FUNCTION
# ══════════════════════════════════════════════════════════════════════════════
def build_dataset(data_dir: str, output_dir: str) -> None:
    data_path = Path(data_dir)
    out_path  = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    npy_files = sorted(data_path.glob("*.npy"))
    if not npy_files:
        raise FileNotFoundError(f"No .npy files in {data_dir}\nCheck DATA_DIR path above.")

    registry: dict[str, int] = {}
    parsed:   list            = []

    for f in npy_files:
        result = parse_filename(f.name)
        if result is None:
            continue
        group, subj_num, stage = result
        key = f"{group}_{subj_num:02d}"
        if key not in registry:
            registry[key] = len(registry)
        parsed.append((f, group, subj_num, stage, registry[key]))

    print(f"\n{'═'*60}")
    print(f"Subjects found : {len(registry)}")
    print(f"Stage files    : {len(parsed)}")
    print(f"Registry       : {list(registry.keys())}")
    print(f"{'═'*60}\n")

    all_X, all_ys, all_yg, all_sid = [], [], [], []
    counts: dict[str, dict[str, int]] = defaultdict(dict)

    for npy_file, group, subj_num, stage, subj_idx in parsed:
        print(f"  Processing: {npy_file.name}", end=" ... ", flush=True)
        X, ys, yg, sid = load_and_extract(str(npy_file), group, subj_num, stage, subj_idx)
        if X.shape[0] == 0:
            print("SKIPPED (0 valid epochs)")
            continue
        all_X.append(X);  all_ys.append(ys)
        all_yg.append(yg); all_sid.append(sid)
        key = f"{group}_{subj_num:02d}"
        counts[key][stage] = int(X.shape[0])
        print(f"{X.shape[0]} epochs")

    X_fin   = np.concatenate(all_X).astype(np.float32)
    ys_fin  = np.concatenate(all_ys).astype(np.int32)
    yg_fin  = np.concatenate(all_yg).astype(np.int32)
    sid_fin = np.concatenate(all_sid).astype(np.int32)

    n = X_fin.shape[0]
    assert ys_fin.shape[0] == yg_fin.shape[0] == sid_fin.shape[0] == n, \
        "Row count mismatch — logic error in load_and_extract"
    assert np.isnan(X_fin).sum() == 0, \
        "NaN in final matrix — a feature extractor returned NaN silently"

    np.save(out_path / "X.npy",           X_fin)
    np.save(out_path / "y_stage.npy",     ys_fin)
    np.save(out_path / "y_group.npy",     yg_fin)
    np.save(out_path / "subject_ids.npy", sid_fin)

    stage_dist = {str(k): int(v) for k, v in zip(*np.unique(ys_fin, return_counts=True))}
    group_dist = {str(k): int(v) for k, v in zip(*np.unique(yg_fin, return_counts=True))}

    metadata = {
        "n_epochs":    n,
        "n_features":  int(X_fin.shape[1]),
        "n_subjects":  len(registry),
        "feature_names": FEATURE_NAMES,
        "subject_registry": registry,
        "epoch_counts_per_subject": {k: dict(v) for k, v in counts.items()},
        "stage_map": STAGE_MAP,
        "group_map": GROUP_MAP,
        "class_distribution_stage": stage_dist,
        "class_distribution_group": group_dist,
    }
    with open(out_path / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"\n{'═'*60}")
    print(f"DATASET SAVED TO: {out_path.resolve()}")
    print(f"  X.npy            : {X_fin.shape}  (dtype={X_fin.dtype})")
    print(f"  y_stage.npy      : {ys_fin.shape}  unique={np.unique(ys_fin)}")
    print(f"  y_group.npy      : {yg_fin.shape}  unique={np.unique(yg_fin)}")
    print(f"  subject_ids.npy  : {sid_fin.shape}  unique subjects={len(np.unique(sid_fin))}")
    print(f"  Stage distribution : {stage_dist}")
    print(f"  Group distribution : {group_dist}")
    print(f"  NaN count          : {np.isnan(X_fin).sum()}   ← must be 0")
    print(f"{'═'*60}")

    print(f"\n{'─'*60}")
    print(f"{'Subject':<22} {'Wake':>6} {'N1':>6} {'N2':>6} {'N3':>6} {'REM':>6}  Group")
    print(f"{'─'*60}")
    stage_cols = ["Wake", "Stage_1", "Stage_2", "Stage_3", "REM"]
    for key, idx in sorted(registry.items(), key=lambda x: x[1]):
        grp   = "Insomnia" if GROUP_MAP.get(key.split('_')[0], 0) == 1 else "Normal"
        row   = counts.get(key, {})
        cells = [str(row.get(s, 0)).rjust(6) for s in stage_cols]
        print(f"  {key:<20} {'  '.join(cells)}  {grp}")
    print(f"{'─'*60}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION F — SMOKE TEST
# ══════════════════════════════════════════════════════════════════════════════
print("Running smoke test on synthetic epoch...")
np.random.seed(42)
_dummy = np.random.randn(17, N_SAMPLES).astype(np.float32) * 50e-6
_fv    = extract_features(_dummy)
assert _fv.shape == (108,),        f"Expected (108,), got {_fv.shape}"
assert np.isnan(_fv).sum() == 0,   f"NaN in smoke-test features"
assert np.isinf(_fv).sum() == 0,   f"Inf in smoke-test features"
print(f"  Smoke test PASSED — feature shape={_fv.shape}, NaNs=0\n")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION G — RUN
# ══════════════════════════════════════════════════════════════════════════════
build_dataset(data_dir=DATA_DIR, output_dir=OUTPUT_DIR)

Running smoke test on synthetic epoch...
  Smoke test PASSED — feature shape=(108,), NaNs=0


════════════════════════════════════════════════════════════
Subjects found : 22
Stage files    : 107
Registry       : ['Insomnia_01', 'Insomnia_03', 'Insomnia_04', 'Insomnia_05', 'Insomnia_06', 'Insomnia_07', 'Insomnia_08', 'Insomnia_09', 'Insomnia_10', 'Insomnia_11', 'Normal_01', 'Normal_02', 'Normal_03', 'Normal_04', 'Normal_05', 'Normal_06', 'Normal_07', 'Normal_08', 'Normal_09', 'Normal_10', 'Normal_11', 'insomaniac_02']
════════════════════════════════════════════════════════════

  Processing: Insomnia_01_REM.npy ... 44 epochs
  Processing: Insomnia_01_Stage_1.npy ... 191 epochs
  Processing: Insomnia_01_Stage_2.npy ... 103 epochs
  Processing: Insomnia_01_Stage_3.npy ... 21 epochs
  Processing: Insomnia_01_Wake.npy ... 421 epochs
  Processing: Insomnia_03_REM.npy ... 14 epochs
  Processing: Insomnia_03_Stage_1.npy ... 376 epochs
  Processing: Insomnia_03_Stage_2.npy ... 127 epochs
  Pr

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE-CELL DATA INSPECTION + NORMALIZATION AUDIT (SHARMA 2021 FEATURES)
# ═══════════════════════════════════════════════════════════════════════════════
import numpy as np
import json
import warnings
from pathlib import Path

# ── scipy for stats ────────────────────────────────────────────────────────────
from scipy import stats as sp_stats

# ── matplotlib / seaborn ───────────────────────────────────────────────────────
try:
    import matplotlib
    matplotlib.use('Agg')           
    import matplotlib.pyplot as plt
    HAS_PLOT = True
except ImportError:
    HAS_PLOT = False
    warnings.warn("matplotlib missing — skipping plots. pip install matplotlib")

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════
ML_DIR     = "/kaggle/working/ml_ready_sharma"
OUTPUT_DIR = "/kaggle/working/inspection_sharma"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

STAGE_NAMES = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
GROUP_NAMES = {0: "Normal", 1: "Insomnia"}

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — LOAD
# ══════════════════════════════════════════════════════════════════════════════
print("Loading Sharma Dataset...")
p           = Path(ML_DIR)
X           = np.load(p / "X.npy")
y_stage     = np.load(p / "y_stage.npy")
y_group     = np.load(p / "y_group.npy")
subject_ids = np.load(p / "subject_ids.npy")

with open(p / "metadata.json") as f:
    meta = json.load(f)

feature_names    = meta["feature_names"]
subject_registry = meta["subject_registry"]          
id_to_name       = {v: k for k, v in subject_registry.items()}

N, D = X.shape
print(f"X shape         : {X.shape}   (N={N} epochs, D={D} features)")
print(f"Unique subjects : {len(np.unique(subject_ids))}")

assert D == 108, f"Expected 108 Sharma features, got {D}"

# Dynamically map feature blocks by Norm type for the Sharma dataset
FEATURE_BLOCKS = {
    "L1 Norms":   [i for i, name in enumerate(feature_names) if "_L1_" in name],
    "L2 Norms":   [i for i, name in enumerate(feature_names) if "_L2_" in name],
    "Linf Norms": [i for i, name in enumerate(feature_names) if "_Linf_" in name],
}

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — NaN / Inf / Zero-Variance AUDIT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 2: Pathological Value Audit")
print(f"{'═'*60}")

nan_per_feat = np.isnan(X).sum(axis=0)
inf_per_feat = np.isinf(X).sum(axis=0)
var_per_feat = np.var(X, axis=0)

n_nan_feats  = (nan_per_feat > 0).sum()
n_inf_feats  = (inf_per_feat > 0).sum()
n_zero_var   = (var_per_feat < 1e-12).sum()

print(f"Features with any NaN         : {n_nan_feats}  (must be 0)")
print(f"Features with any Inf         : {n_inf_feats}  (must be 0)")
print(f"Features with zero variance   : {n_zero_var}  (useless for ML — drop these)")

if n_zero_var > 0:
    zv_names = [feature_names[i] for i in np.where(var_per_feat < 1e-12)[0]]
    print(f"  Zero-variance features: {zv_names}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — DISTRIBUTION SHAPE AUDIT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 3: Distribution Shape — Normality Screening")
print(f"{'═'*60}")

rng = np.random.default_rng(42)
sample_idx = rng.choice(N, size=min(2000, N), replace=False)
X_sample   = X[sample_idx]

non_normal_count = 0
skew_vals, kurt_vals = [], []

for i in range(D):
    col = X_sample[:, i]
    skew_vals.append(sp_stats.skew(col))
    kurt_vals.append(sp_stats.kurtosis(col))     
    _, p = sp_stats.normaltest(col)               
    if p < 0.05:
        non_normal_count += 1

skew_arr = np.array(skew_vals)
kurt_arr = np.array(kurt_vals)

print(f"Non-normal features (D'Agostino k², α=0.05) : {non_normal_count}/{D}")
print(f"Skewness  — mean={skew_arr.mean():.2f}, max_abs={np.abs(skew_arr).max():.2f}")

print(f"\nPer norm-block skewness profile:")
for block_name, indices in FEATURE_BLOCKS.items():
    blk_skew = np.abs(skew_arr[indices])
    print(f"  {block_name:<20}: mean|skew|={blk_skew.mean():.2f}, max|skew|={blk_skew.max():.2f}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — DYNAMIC RANGE AUDIT
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 4: Dynamic Range per Feature Block")
print(f"{'═'*60}")
print(f"{'Block':<22} {'Min':>12} {'Max':>12} {'Mean':>12} {'Std':>12}")
print(f"{'─'*72}")

for block_name, indices in FEATURE_BLOCKS.items():
    blk = X[:, indices]
    print(f"  {block_name:<20} {blk.min():>12.4e} {blk.max():>12.4e} "
          f"{blk.mean():>12.4e} {blk.std():>12.4e}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — CLASS SEPARABILITY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 5: Top Discriminative Features — Cohen's d")
print(f"{'═'*60}")

X_ins = X[y_group == 1]
X_nor = X[y_group == 0]

mean_ins = X_ins.mean(axis=0)
mean_nor = X_nor.mean(axis=0)

std_pool = np.sqrt(
    (np.var(X_ins, axis=0) * (len(X_ins) - 1) +
     np.var(X_nor, axis=0) * (len(X_nor) - 1)) /
    (len(X_ins) + len(X_nor) - 2 + 1e-12))

cohens_d = (mean_ins - mean_nor) / (std_pool + 1e-12)

top15_idx = np.argsort(np.abs(cohens_d))[::-1][:15]

print(f"{'Rank':<5} {'Feature':<40} {'Cohen d':>9}")
print(f"{'─'*60}")
for rank, idx in enumerate(top15_idx):
    print(f"  {rank+1:<4} {feature_names[idx]:<40} {cohens_d[idx]:>9.3f}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 6 — PLOTS (saved to disk)
# ══════════════════════════════════════════════════════════════════════════════
if HAS_PLOT:
    print(f"\n{'═'*60}")
    print("SECTION 6: Generating Plots → saving to", OUTPUT_DIR)
    print(f"{'═'*60}")

    # Plot 1: Skewness distribution per Norm block
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, (block_name, indices) in zip(axes, FEATURE_BLOCKS.items()):
        ax.hist(skew_arr[indices], bins=15, edgecolor='black', color='steelblue', alpha=0.8)
        ax.axvline(0,  color='green',  lw=1.5, linestyle='--')
        ax.axvline(+1.5, color='orange', lw=1.0, linestyle=':')
        ax.set_title(block_name)
        ax.set_xlabel("Skewness")
    plt.suptitle("Skewness Distribution per Block", y=1.05)
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot1_skewness_sharma.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    # Plot 2: Cohen's d bar chart
    top20_idx = np.argsort(np.abs(cohens_d))[::-1][:20]
    top20_d   = cohens_d[top20_idx]
    top20_nm  = [feature_names[i] for i in top20_idx]
    colors    = ['tomato' if d > 0 else 'steelblue' for d in top20_d]

    fig, ax = plt.subplots(figsize=(14, 6))
    ax.bar(range(20), top20_d, color=colors, edgecolor='black', alpha=0.85)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks(range(20))
    ax.set_xticklabels(top20_nm, fontsize=8, rotation=45, ha='right')
    ax.set_ylabel("Cohen's d (Insomnia − Normal)")
    ax.set_title("Top 20 Wavelet Features by Effect Size")
    plt.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/plot2_cohens_d_sharma.png", dpi=150, bbox_inches='tight')
    plt.close(fig)

    print("  Saved plots successfully.")
else:
    print("Plots skipped — matplotlib not available.")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION 7 — NORMALIZATION RECOMMENDATION (SHARMA)
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*60}")
print("SECTION 7: Normalization Recommendation for Sharma Features")
print(f"{'═'*60}")
print("""
Sharma's features (L1, L2, Linf norms) represent raw amplitude scales and will 
exhibit extreme dynamic ranges and heavy right-skews (power-law distribution).

MANDATORY PREPROCESSING PIPELINE:
1. Apply np.log1p() to ALL 108 features. Since wavelet norms are strictly positive, 
   clipping at 0 before log1p is mathematically safe and necessary to compress the scale.
2. Apply RobustScaler() inside your LOSO cross-validation loop. Do NOT use 
   StandardScaler, as the L-infinity norm (peak amplitude) will be highly sensitive 
   to single-epoch motion artifacts.
""")

Loading Sharma Dataset...
X shape         : (13007, 108)   (N=13007 epochs, D=108 features)
Unique subjects : 22

════════════════════════════════════════════════════════════
SECTION 2: Pathological Value Audit
════════════════════════════════════════════════════════════
Features with any NaN         : 0  (must be 0)
Features with any Inf         : 0  (must be 0)
Features with zero variance   : 6  (useless for ML — drop these)
  Zero-variance features: ['wvl_D1_Linf_ch0', 'wvl_D1_Linf_ch1', 'wvl_D1_Linf_ch2', 'wvl_D1_Linf_ch3', 'wvl_D1_Linf_ch4', 'wvl_D1_Linf_ch5']

════════════════════════════════════════════════════════════
SECTION 3: Distribution Shape — Normality Screening
════════════════════════════════════════════════════════════
Non-normal features (D'Agostino k², α=0.05) : 108/108
Skewness  — mean=10.67, max_abs=27.47

Per norm-block skewness profile:
  L1 Norms            : mean|skew|=12.30, max|skew|=27.47
  L2 Norms            : mean|skew|=10.87, max|skew|=22.77
  Linf Norm

In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# 31-COMBINATION POOLED 10-FOLD CV — SHARMA TABLE 7 REPLICATION + EXTENSION
#
# ⚠ EXPLICIT LEAKAGE WARNING:
#   This uses POOLED stratified 10-fold CV on epochs.
#   Subject epochs from the same person CAN appear in both train and test folds.
#   This EXACTLY replicates Sharma 2021 methodology (Table 7).
#   Results are an UPPER BOUND — do NOT claim subject-independent generalization.
#   For honest generalization, use the LOSO pipeline.
#
# OUTPUT: Matches Sharma Table 7 columns:
#   Accuracy | Precision | Specificity | Sensitivity | F1 | Kappa | AROC
#   Positive class = Insomnia (label=1)
#
# RUNTIME ESTIMATE (Kaggle CPU):
#   LogReg/KNN/DT/RF/XGB  ≈ 20–40 min total
#   SVM (probability=True) ≈ +30–60 min — toggle SKIP_SVM=True to skip
# ═══════════════════════════════════════════════════════════════════════════════

# ── 0. DEPENDENCIES ────────────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["xgboost", "scikit-learn", "numpy", "scipy", "pandas"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# ── 1. IMPORTS ─────────────────────────────────────────────────────────────────
import json
import time
import warnings
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.dummy              import DummyClassifier
from sklearn.linear_model       import LogisticRegression
from sklearn.neighbors          import KNeighborsClassifier
from sklearn.svm                import SVC
from sklearn.tree               import DecisionTreeClassifier
from sklearn.ensemble           import RandomForestClassifier
from sklearn.preprocessing      import RobustScaler
from sklearn.feature_selection  import VarianceThreshold
from sklearn.base               import clone as sk_clone
from sklearn.model_selection    import StratifiedKFold
from sklearn.metrics            import (
    accuracy_score, cohen_kappa_score, f1_score,
    precision_score, recall_score, roc_auc_score, confusion_matrix
)

from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — edit only this block
# ══════════════════════════════════════════════════════════════════════════════
ML_DIR     = "/kaggle/working/ml_ready_sharma"
OUTPUT_DIR = "/kaggle/working/sharma_31combo_results"
SEED       = 42
N_FOLDS    = 10
SKIP_SVM   = True   # ← Set True to skip SVM and save ~30–60 min

np.random.seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# Stage index → label mapping
STAGE_MAP  = {0: "W", 1: "N1", 2: "N2", 3: "N3", 4: "R"}
GROUP_NAMES = {0: "Normal", 1: "Insomnia"}

# Sharma-specific named combinations (for alias display)
SHARMA_ALIASES = {
    (1, 2):       "LSS",
    (1, 2, 3):    "NREM",
    (0, 1, 2, 3, 4): "ALL",
}

# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD & PREPROCESS
# ══════════════════════════════════════════════════════════════════════════════
print("=" * 70)
print("SECTION A — Loading and Preprocessing")
print("=" * 70)

p           = Path(ML_DIR)
X_raw       = np.load(p / "X.npy")
y_stage     = np.load(p / "y_stage.npy")
y_group     = np.load(p / "y_group.npy")
subject_ids = np.load(p / "subject_ids.npy")

with open(p / "metadata.json") as f:
    meta = json.load(f)

feature_names    = np.array(meta["feature_names"])
subject_registry = meta["subject_registry"]

print(f"  X_raw       : {X_raw.shape}  (epochs × features)")
print(f"  y_stage     : {np.unique(y_stage, return_counts=True)}")
print(f"  y_group     : {np.unique(y_group, return_counts=True)}")
print(f"  Subjects    : {len(np.unique(subject_ids))}")
assert X_raw.shape[1] == 108, f"Expected 108 features, got {X_raw.shape[1]}"

# Global log1p transform — wavelet norms are positive, right-skewed
X_t = np.log1p(np.clip(X_raw.astype(np.float64), 0, None))
assert np.isnan(X_t).sum() == 0 and np.isinf(X_t).sum() == 0, \
    "NaN/Inf after log1p — check upstream extraction"

# Drop zero-variance features
vt         = VarianceThreshold(threshold=1e-6)
X_filtered = vt.fit_transform(X_t).astype(np.float32)
kept_names = feature_names[vt.get_support()]
n_dropped  = (~vt.get_support()).sum()

print(f"  Zero-variance dropped : {n_dropped}  (expected 6 × D1_Linf)")
print(f"  Features remaining    : {X_filtered.shape[1]}  (expected 102)")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — GENERATE ALL 31 STAGE COMBINATIONS
# ══════════════════════════════════════════════════════════════════════════════
all_combos: list[tuple[int, ...]] = []
for r in range(1, 6):
    for combo in itertools.combinations([0, 1, 2, 3, 4], r):
        all_combos.append(combo)

assert len(all_combos) == 31, f"Expected 31 combos, got {len(all_combos)}"

def combo_label(combo: tuple) -> str:
    """Human-readable stage combination label."""
    alias = SHARMA_ALIASES.get(combo)
    name  = "+".join(STAGE_MAP[s] for s in combo)
    return f"{name} ({alias})" if alias else name

def combo_epoch_counts(combo: tuple) -> dict:
    """Returns epoch count per group for a given stage combination."""
    mask    = np.isin(y_stage, list(combo))
    y_sub   = y_group[mask]
    ins_n   = int((y_sub == 1).sum())
    norm_n  = int((y_sub == 0).sum())
    return {"total": ins_n + norm_n, "insomnia": ins_n, "normal": norm_n}

print(f"\nSECTION B — {len(all_combos)} stage combinations generated")
for c in all_combos:
    counts = combo_epoch_counts(c)
    print(f"  {combo_label(c):<32}  "
          f"total={counts['total']:>5}  "
          f"Ins={counts['insomnia']:>4}  Nor={counts['normal']:>4}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION C — MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════
def get_models(seed: int = SEED, skip_svm: bool = SKIP_SVM) -> dict:
    """
    Returns classifier dict. SVM has probability=True for AUROC.
    All classifiers use class_weight='balanced' to handle epoch imbalance.
    """
    models = {
        "Dummy":        DummyClassifier(strategy="most_frequent", random_state=seed),
        "LogReg":       LogisticRegression(
                            C=1.0, max_iter=3000, solver="saga",
                            class_weight="balanced", n_jobs=-1, random_state=seed),
        "KNN_k7":       KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
        "DecisionTree": DecisionTreeClassifier(
                            max_depth=10, min_samples_leaf=10,
                            class_weight="balanced", random_state=seed),
        "RandomForest": RandomForestClassifier(
                            n_estimators=300, max_depth=12, min_samples_leaf=5,
                            class_weight="balanced", n_jobs=-1, random_state=seed),
        "XGBoost":      XGBClassifier(
                            n_estimators=300, max_depth=6, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.8,
                            objective="binary:logistic", eval_metric="logloss",
                            use_label_encoder=False,
                            n_jobs=-1, random_state=seed, verbosity=0),
    }
    if not skip_svm:
        models["SVM_RBF"] = SVC(
            C=1.0, kernel="rbf", gamma="scale",
            class_weight="balanced",
            probability=True,         # Required for AUROC
            random_state=seed)
    else:
        print("  [INFO] SVM skipped (SKIP_SVM=True)")
    return models

# ══════════════════════════════════════════════════════════════════════════════
# SECTION D — METRIC HELPER
# ══════════════════════════════════════════════════════════════════════════════
def compute_metrics(y_true: np.ndarray,
                    y_pred: np.ndarray,
                    y_prob: np.ndarray | None) -> dict:
    """
    Computes Sharma Table 7 metrics. Positive class = Insomnia (1).

    Specificity = TN / (TN + FP)  = Recall of Normal class (0)
    Sensitivity = TP / (TP + FN)  = Recall of Insomnia class (1)
    Precision   = TP / (TP + FP)  for Insomnia class

    Returns dict with keys:
        accuracy, precision, specificity, sensitivity, f1, kappa, auroc
    """
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel() if cm.shape == (2, 2) else (0, 0, 0, 0)

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    auroc = 0.5
    if y_prob is not None and len(np.unique(y_true)) > 1:
        try:
            auroc = float(roc_auc_score(y_true, y_prob))
        except Exception:
            auroc = 0.5

    return {
        "accuracy":    float(accuracy_score(y_true, y_pred)),
        "precision":   float(precision_score(y_true, y_pred,
                             pos_label=1, zero_division=0)),
        "specificity": float(specificity),
        "sensitivity": float(sensitivity),
        "f1":          float(f1_score(y_true, y_pred,
                             pos_label=1, zero_division=0)),
        "kappa":       float(cohen_kappa_score(y_true, y_pred)
                             if len(np.unique(y_true)) > 1 else 0.0),
        "auroc":       auroc,
    }

# ══════════════════════════════════════════════════════════════════════════════
# SECTION E — STRATIFIED 10-FOLD CV ENGINE
# ══════════════════════════════════════════════════════════════════════════════
def run_10fold_cv(
    X:          np.ndarray,
    y:          np.ndarray,
    model_name: str,
    model,
    seed:       int = SEED,
) -> dict:
    """
    Runs stratified 10-fold CV for a single (combo, classifier) pair.

    I/O:
        X : (N, 102)  float32 — already log1p transformed, NOT yet scaled
        y : (N,)      int     — binary: 0=Normal, 1=Insomnia

    Returns: dict of mean ± std for all Sharma metrics.
    """
    skf    = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_metrics: list[dict] = []

    for fold_i, (train_idx, test_idx) in enumerate(skf.split(X, y)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Fit scaler on train only — no leakage within fold
        scaler   = RobustScaler()
        X_tr_sc  = scaler.fit_transform(X_train)
        X_te_sc  = scaler.transform(X_test)

        clf = sk_clone(model)
        try:
            clf.fit(X_tr_sc, y_train)
        except Exception as e:
            print(f"      [WARN] {model_name} fold {fold_i} fit failed: {e}")
            continue

        y_pred = clf.predict(X_te_sc)

        # Get probability for AUROC
        y_prob = None
        if hasattr(clf, "predict_proba"):
            try:
                y_prob = clf.predict_proba(X_te_sc)[:, 1]
            except Exception:
                pass
        elif hasattr(clf, "decision_function"):
            try:
                y_prob = clf.decision_function(X_te_sc)
            except Exception:
                pass

        m = compute_metrics(y_test, y_pred, y_prob)
        fold_metrics.append(m)

    if not fold_metrics:
        return {}

    # Aggregate across folds
    agg = {}
    for key in fold_metrics[0]:
        vals        = [fm[key] for fm in fold_metrics]
        agg[f"{key}_mean"] = float(np.mean(vals))
        agg[f"{key}_std"]  = float(np.std(vals))

    return agg

# ══════════════════════════════════════════════════════════════════════════════
# SECTION F — MAIN LOOP: 31 combinations × N classifiers
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SECTION F — Running 31 × N_classifiers 10-fold CV")
print(f"  ⚠ LEAKAGE WARNING: Pooled 10-fold — replicates Sharma, NOT LOSO")
print("=" * 70)

models     = get_models()
model_names = list(models.keys())
n_models    = len(model_names)

# Master results: results[combo_label][model_name] = agg_dict
master_results: dict[str, dict] = {}

total_runs = len(all_combos) * n_models
run_count  = 0
t_global   = time.time()

for combo in all_combos:
    label  = combo_label(combo)
    mask   = np.isin(y_stage, list(combo))
    X_sub  = X_filtered[mask]
    y_sub  = y_group[mask]
    counts = combo_epoch_counts(combo)

    # Skip combos with fewer than N_FOLDS samples per class
    if counts["insomnia"] < N_FOLDS or counts["normal"] < N_FOLDS:
        print(f"\n  [SKIP] {label} — too few epochs for {N_FOLDS}-fold "
              f"(Ins={counts['insomnia']}, Nor={counts['normal']})")
        continue

    print(f"\n  {'─'*68}")
    print(f"  COMBO: {label:<32}  "
          f"N={counts['total']}  Ins={counts['insomnia']}  Nor={counts['normal']}")
    print(f"  {'─'*68}")

    master_results[label] = {"_meta": counts}

    for model_name, model in models.items():
        run_count += 1
        t0 = time.time()
        result = run_10fold_cv(X_sub, y_sub, model_name, model)
        elapsed = time.time() - t0

        master_results[label][model_name] = result

        if result:
            print(f"    {model_name:<14}  "
                  f"acc={result['accuracy_mean']:.4f}±{result['accuracy_std']:.3f}  "
                  f"sens={result['sensitivity_mean']:.4f}  "
                  f"spec={result['specificity_mean']:.4f}  "
                  f"f1={result['f1_mean']:.4f}  "
                  f"kappa={result['kappa_mean']:.4f}  "
                  f"auc={result['auroc_mean']:.4f}  "
                  f"[{elapsed:.1f}s]  "
                  f"({run_count}/{total_runs})")
        else:
            print(f"    {model_name:<14}  [FAILED]")

elapsed_total = time.time() - t_global
print(f"\n  Total runtime: {elapsed_total/60:.1f} min")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION G — BUILD SHARMA-STYLE SUMMARY TABLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("SECTION G — Table 7 Style Summary")
print("=" * 70)

METRIC_COLS = ["accuracy", "precision", "specificity", "sensitivity",
               "f1", "kappa", "auroc"]

def pct(v): return f"{v*100:.2f}%"
def fmt(v): return f"{v:.4f}"

rows = []
for label, combo_data in master_results.items():
    meta = combo_data.get("_meta", {})
    for model_name in model_names:
        res = combo_data.get(model_name, {})
        if not res:
            continue
        row = {
            "Stage Combo":  label,
            "N Epochs":     meta.get("total", "?"),
            "Model":        model_name,
        }
        for m in METRIC_COLS:
            row[m.capitalize()]      = res.get(f"{m}_mean", np.nan)
            row[f"{m.capitalize()}_std"] = res.get(f"{m}_std", np.nan)
        rows.append(row)

df = pd.DataFrame(rows)

# ── Print Sharma-style table per model ────────────────────────────────────────
for model_name in model_names:
    sub = df[df["Model"] == model_name].copy()
    if sub.empty:
        continue

    print(f"\n  ── {model_name} ──")
    print(f"  {'Stage Combo':<30} {'N':>6}  "
          f"{'Acc(%)':>8} {'Prec':>6} {'Spec':>6} {'Sens':>6} "
          f"{'F1':>6} {'Kappa':>7} {'AROC':>6}")
    print(f"  {'─'*90}")

    for _, row in sub.iterrows():
        print(f"  {row['Stage Combo']:<30} {row['N Epochs']:>6}  "
              f"{row['Accuracy']*100:>7.2f}% "
              f"{row['Precision']:>6.4f} "
              f"{row['Specificity']:>6.4f} "
              f"{row['Sensitivity']:>6.4f} "
              f"{row['F1']:>6.4f} "
              f"{row['Kappa']:>7.4f} "
              f"{row['Auroc']:>6.4f}")

# ── Find best combo per model ──────────────────────────────────────────────────
print(f"\n  {'─'*70}")
print("  BEST COMBINATION per model (ranked by Kappa)")
print(f"  {'─'*70}")
print(f"  {'Model':<14}  {'Best Combo':<32}  "
      f"{'Kappa':>7}  {'AUROC':>7}  {'Sens':>6}  {'Spec':>6}")
print(f"  {'─'*70}")

for model_name in model_names:
    sub = df[df["Model"] == model_name]
    if sub.empty or sub["Kappa"].isna().all():
        continue
    best = sub.loc[sub["Kappa"].idxmax()]
    print(f"  {model_name:<14}  {best['Stage Combo']:<32}  "
          f"{best['Kappa']:>7.4f}  {best['Auroc']:>7.4f}  "
          f"{best['Sensitivity']:>6.4f}  {best['Specificity']:>6.4f}")

# ── Best model × best combo overall ───────────────────────────────────────────
non_dummy = df[df["Model"] != "Dummy"]
if not non_dummy.empty and not non_dummy["Kappa"].isna().all():
    global_best = non_dummy.loc[non_dummy["Kappa"].idxmax()]
    print(f"\n  ══ GLOBAL BEST (Kappa) ══")
    print(f"  Model     : {global_best['Model']}")
    print(f"  Combo     : {global_best['Stage Combo']}")
    print(f"  Accuracy  : {global_best['Accuracy']*100:.2f}%")
    print(f"  Precision : {global_best['Precision']:.4f}")
    print(f"  Specificity: {global_best['Specificity']:.4f}")
    print(f"  Sensitivity: {global_best['Sensitivity']:.4f}")
    print(f"  F1-Score  : {global_best['F1']:.4f}")
    print(f"  Kappa     : {global_best['Kappa']:.4f}")
    print(f"  AUROC     : {global_best['Auroc']:.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION H — SAVE ALL OUTPUTS
# ══════════════════════════════════════════════════════════════════════════════
out = Path(OUTPUT_DIR)

# 1. Master CSV — all combinations × all models
csv_path = out / "sharma_31combo_all_results.csv"
df.to_csv(csv_path, index=False)
print(f"\n  Master CSV saved  : {csv_path}")

# 2. Per-model CSVs matching Sharma Table 7 layout
for model_name in model_names:
    sub = df[df["Model"] == model_name].copy()
    if sub.empty:
        continue
    sub_path = out / f"table7_{model_name}.csv"
    sub.to_csv(sub_path, index=False)

# 3. JSON for downstream analysis
def _jsonify(obj):
    if isinstance(obj, (np.integer,)):  return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray):     return obj.tolist()
    return obj

with open(out / "sharma_31combo_master.json", "w") as f:
    json.dump(
        {k: {m: {mk: _jsonify(mv) for mk, mv in mv2.items()}
             for m, mv2 in v.items() if isinstance(mv2, dict)}
         for k, v in master_results.items()},
        f, indent=2
    )

# 4. Best-per-model summary CSV
summary_rows = []
for model_name in model_names:
    sub = df[df["Model"] == model_name]
    if sub.empty or sub["Kappa"].isna().all():
        continue
    best = sub.loc[sub["Kappa"].idxmax()].to_dict()
    summary_rows.append(best)

pd.DataFrame(summary_rows).to_csv(out / "best_per_model_summary.csv", index=False)

print(f"  Per-model CSVs    : {OUTPUT_DIR}/table7_<model>.csv")
print(f"  Master JSON       : {OUTPUT_DIR}/sharma_31combo_master.json")
print(f"  Best summary CSV  : {OUTPUT_DIR}/best_per_model_summary.csv")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION I — INTERPRETATION GUARD
# ══════════════════════════════════════════════════════════════════════════════
print(f"""
{'═'*70}
INTERPRETATION GUARD — READ BEFORE REPORTING
{'═'*70}
  ⚠ These results use POOLED 10-fold CV (Sharma 2021 methodology).
  ⚠ Subject leakage IS present. This is expected — you are replicating Sharma.
  ⚠ DO NOT report these numbers as "subject-independent" or "generalizable".
  ⚠ Compare against your LOSO results to quantify the leakage inflation gap.

  REPORTING TEMPLATE:
    "Following Sharma et al. (2021), a pooled stratified 10-fold CV was used
     to replicate the experimental setup. Subject-independent performance was
     assessed separately using Leave-One-Subject-Out (LOSO) cross-validation."

  KAPPA THRESHOLDS (Landis & Koch, 1977):
    < 0.20  = Slight     (do not report)
    0.21–0.40 = Fair
    0.41–0.60 = Moderate
    0.61–0.80 = Substantial   ← publication threshold
    > 0.80  = Near-perfect
{'═'*70}
""")

SECTION A — Loading and Preprocessing
  X_raw       : (13007, 108)  (epochs × features)
  y_stage     : (array([0, 1, 2, 3, 4], dtype=int32), array([3098, 4605, 2272, 2052,  980]))
  y_group     : (array([0, 1], dtype=int32), array([6705, 6302]))
  Subjects    : 22
  Zero-variance dropped : 84  (expected 6 × D1_Linf)
  Features remaining    : 24  (expected 102)

SECTION B — 31 stage combinations generated
  W                                 total= 3098  Ins=2165  Nor= 933
  N1                                total= 4605  Ins=2065  Nor=2540
  N2                                total= 2272  Ins= 736  Nor=1536
  N3                                total= 2052  Ins= 759  Nor=1293
  R                                 total=  980  Ins= 577  Nor= 403
  W+N1                              total= 7703  Ins=4230  Nor=3473
  W+N2                              total= 5370  Ins=2901  Nor=2469
  W+N3                              total= 5150  Ins=2924  Nor=2226
  W+R                               total= 4078

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# SINGLE-CELL LOSO CLASSIFICATION PIPELINE — SHARMA 2021 WAVELET FEATURES
# ═══════════════════════════════════════════════════════════════════════════════

# ── 0. INSTALL DEPENDENCIES ────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["xgboost", "scikit-learn", "numpy", "scipy"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

# ── 1. IMPORTS ─────────────────────────────────────────────────────────────────
import json
import time
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict

from sklearn.dummy             import DummyClassifier
from sklearn.linear_model      import LogisticRegression
from sklearn.neighbors         import KNeighborsClassifier
from sklearn.svm               import SVC
from sklearn.tree              import DecisionTreeClassifier
from sklearn.ensemble          import RandomForestClassifier
from sklearn.preprocessing     import RobustScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.base              import clone as sk_clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics           import (accuracy_score, cohen_kappa_score,
                                       f1_score, classification_report,
                                       confusion_matrix)
from scipy.stats               import wilcoxon
from xgboost                   import XGBClassifier

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# CONFIG — only edit this block
# ══════════════════════════════════════════════════════════════════════════════
ML_DIR     = "/kaggle/working/ml_ready_sharma"
OUTPUT_DIR = "/kaggle/working/loso_results_sharma"
SEED       = 42
np.random.seed(SEED)

STAGE_NAMES = {0: "Wake", 1: "N1", 2: "N2", 3: "N3", 4: "REM"}
GROUP_NAMES = {0: "Normal", 1: "Insomnia"}

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION A — LOAD DATA
# ══════════════════════════════════════════════════════════════════════════════
print("Loading Sharma wavelet dataset...")
p           = Path(ML_DIR)
X_raw       = np.load(p / "X.npy")
y_stage     = np.load(p / "y_stage.npy")
y_group     = np.load(p / "y_group.npy")
subject_ids = np.load(p / "subject_ids.npy")

with open(p / "metadata.json") as f:
    meta = json.load(f)

feature_names    = np.array(meta["feature_names"])
subject_registry = meta["subject_registry"]
id_to_name       = {v: k for k, v in subject_registry.items()}

print(f"Loaded: X={X_raw.shape}, y_stage={y_stage.shape}, "
      f"subjects={len(np.unique(subject_ids))}")

assert X_raw.shape[1] == 108, f"Expected 108 Sharma features, got {X_raw.shape[1]}"

# ══════════════════════════════════════════════════════════════════════════════
# SECTION B — GLOBAL PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════════
print("\nApplying GLOBAL log1p transform to ALL wavelet features...")
# Wavelet norms are strictly positive, but exhibit severe right-skew.
# Global log1p is mandatory to compress scale before RobustScaler.
X_transformed = X_raw.copy().astype(np.float64)
X_transformed = np.log1p(np.clip(X_transformed, 0, None))

assert np.isnan(X_transformed).sum() == 0, "NaN after log1p"
assert np.isinf(X_transformed).sum() == 0, "Inf after log1p"

# Drop zero-variance features (specifically the 6 D1_Linf channel features)
vt         = VarianceThreshold(threshold=1e-6)
X_filtered = vt.fit_transform(X_transformed).astype(np.float32)
kept_mask  = vt.get_support()
kept_names = feature_names[kept_mask]
n_dropped  = (~kept_mask).sum()

print(f"  Zero-variance features dropped: {n_dropped}  (Expecting 6 from D1_Linf)")
print(f"  Features remaining: {X_filtered.shape[1]}  (Expecting 102)")
print(f"  Final X shape: {X_filtered.shape}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION C — MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════════════════════
def get_models(n_classes: int, seed: int = SEED) -> dict:
    return {
        "Dummy_Majority": DummyClassifier(
            strategy="most_frequent", random_state=seed
        ),
        "LogReg_L2": LogisticRegression(
            C=1.0, max_iter=3000, solver="saga",
            class_weight="balanced",                              
            multi_class="multinomial", n_jobs=-1, random_state=seed
        ),
        "KNN_k7": KNeighborsClassifier(
            n_neighbors=7, metric="euclidean", n_jobs=-1
        ),
        "SVM_RBF": SVC(
            C=1.0, kernel="rbf", gamma="scale",
            class_weight="balanced",                              
            decision_function_shape="ovr", random_state=seed
        ),
        "DecisionTree": DecisionTreeClassifier(
            max_depth=10, min_samples_leaf=10,
            class_weight="balanced",                              
            random_state=seed
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=300, max_depth=12, min_samples_leaf=5,
            class_weight="balanced",                              
            n_jobs=-1, random_state=seed
        ),
        "XGBoost": XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="mlogloss",
            n_jobs=-1, random_state=seed, verbosity=0,
            num_class=n_classes if n_classes > 2 else None,
            objective="multi:softmax" if n_classes > 2 else "binary:logistic"
        ),
    }

# ══════════════════════════════════════════════════════════════════════════════
# SECTION D — LOSO ENGINE
# ══════════════════════════════════════════════════════════════════════════════
def run_loso(
    X: np.ndarray,
    y: np.ndarray,
    subject_ids: np.ndarray,
    task_name: str,
    label_names: dict,
    output_dir: str,
) -> dict:
    unique_subjects = np.unique(subject_ids)
    n_subjects      = len(unique_subjects)
    unique_classes  = np.unique(y)
    n_classes       = len(unique_classes)

    label_remap = {old: new for new, old in enumerate(unique_classes)}
    y_remapped  = np.array([label_remap[yi] for yi in y], dtype=np.int32)
    remap_names = {label_remap[k]: v for k, v in label_names.items()
                   if k in label_remap}

    print(f"\n{'═'*70}")
    print(f"TASK : {task_name}")
    print(f"FOLDS: {n_subjects} (LOSO) | Epochs: {len(y)} | Classes: {n_classes}")
    dist = {remap_names[k]: int(v)
            for k, v in zip(*np.unique(y_remapped, return_counts=True))}
    print(f"Class distribution: {dist}")
    print(f"{'═'*70}")

    models      = get_models(n_classes, seed=SEED)
    all_results = {
        name: {"fold_accs": [], "fold_kappas": [], "fold_f1s": [],
               "fold_subjects": [], "fold_preds": [], "fold_trues": []}
        for name in models
    }
    fold_times = defaultdict(list)

    for fold_i, test_subj in enumerate(unique_subjects):
        test_mask  = subject_ids == test_subj
        train_mask = ~test_mask

        assert not np.any(subject_ids[train_mask] == test_subj), \
            f"DATA LEAKAGE: subject {test_subj} in training fold {fold_i}"

        X_train_raw = X[train_mask]
        X_test_raw  = X[test_mask]
        y_train     = y_remapped[train_mask]
        y_test      = y_remapped[test_mask]

        if len(np.unique(y_train)) < 2:
            print(f"  [SKIP] Fold {fold_i+1}: only 1 class in training set")
            continue

        # Scaling — fit on train only
        scaler     = RobustScaler()
        X_train_sc = scaler.fit_transform(X_train_raw)
        X_test_sc  = scaler.transform(X_test_raw)

        xgb_sample_weights = compute_sample_weight(
            class_weight='balanced', y=y_train
        )

        subj_name = id_to_name[test_subj]
        print(f"\n  Fold {fold_i+1:>2}/{n_subjects} — test={subj_name} "
              f"({y_test.shape[0]} epochs, classes={np.unique(y_test)})")

        for model_name, model in models.items():
            t0  = time.time()
            clf = sk_clone(model)

            try:
                if "XGBoost" in model_name:
                    clf.fit(X_train_sc, y_train,
                            sample_weight=xgb_sample_weights)
                else:
                    clf.fit(X_train_sc, y_train)

                y_pred = clf.predict(X_test_sc)

            except Exception as e:
                print(f"    [{model_name}] FAILED: {e}")
                continue

            elapsed = time.time() - t0
            fold_times[model_name].append(elapsed)

            acc   = accuracy_score(y_test, y_pred)
            kappa = (cohen_kappa_score(y_test, y_pred)
                     if len(np.unique(y_test)) > 1 else 0.0)
            f1    = f1_score(y_test, y_pred, average="macro", zero_division=0)

            all_results[model_name]["fold_accs"].append(acc)
            all_results[model_name]["fold_kappas"].append(kappa)
            all_results[model_name]["fold_f1s"].append(f1)
            all_results[model_name]["fold_subjects"].append(subj_name)
            all_results[model_name]["fold_preds"].extend(y_pred.tolist())
            all_results[model_name]["fold_trues"].extend(y_test.tolist())

            print(f"    {model_name:<18} acc={acc:.3f}  kappa={kappa:.3f}  "
                  f"f1={f1:.3f}  t={elapsed:.1f}s")

    # ── AGGREGATE ─────────────────────────────────────────────────────────────
    dummy_accs = np.array(all_results["Dummy_Majority"]["fold_accs"])

    print(f"\n{'═'*70}")
    print(f"LOSO SUMMARY — {task_name}")
    print(f"{'─'*70}")
    print(f"{'Model':<20} {'Acc':>7} {'±':>5} {'Kappa':>7} {'±':>5} "
          f"{'F1':>7} {'±':>5} {'p_vs_Dummy':>11} {'Sig':>5} {'AvgTime':>9}")
    print(f"{'─'*70}")

    for model_name, res in all_results.items():
        accs   = np.array(res["fold_accs"])
        kappas = np.array(res["fold_kappas"])
        f1s    = np.array(res["fold_f1s"])

        if len(accs) == 0:
            continue

        if model_name != "Dummy_Majority" and len(accs) >= 5:
            common = min(len(accs), len(dummy_accs))
            try:
                _, p_val = wilcoxon(accs[:common], dummy_accs[:common])
            except Exception:
                p_val = float("nan")
        else:
            p_val = float("nan")

        sig = ("***" if p_val < 0.001 else
               "**"  if p_val < 0.01  else
               "*"   if p_val < 0.05  else "ns")

        avg_time = (np.mean(fold_times[model_name])
                    if fold_times[model_name] else 0.0)

        res.update({
            "mean_acc":   float(accs.mean()),   "std_acc":   float(accs.std()),
            "mean_kappa": float(kappas.mean()), "std_kappa": float(kappas.std()),
            "mean_f1":    float(f1s.mean()),    "std_f1":    float(f1s.std()),
            "p_vs_dummy": float(p_val),
            "avg_time_s": float(avg_time),
        })

        print(f"  {model_name:<20} {accs.mean():.3f} ±{accs.std():.3f} "
              f"{kappas.mean():>7.3f} ±{kappas.std():.3f} "
              f"{f1s.mean():>7.3f} ±{f1s.std():.3f} "
              f"{p_val:>11.4f} {sig:>5} {avg_time:>8.1f}s")

    # ── BEST MODEL REPORT ─────────────────────────────────────────────────────
    non_dummy = {k: v for k, v in all_results.items()
                 if k != "Dummy_Majority" and v.get("mean_f1") is not None}
    if non_dummy:
        best_name = max(non_dummy, key=lambda k: non_dummy[k].get("mean_f1", 0))
        best      = all_results[best_name]
        print(f"\n{'─'*70}")
        print(f"BEST MODEL: {best_name}  (macro-F1={best['mean_f1']:.3f}  "
              f"Kappa={best['mean_kappa']:.3f})")
        print(classification_report(
            best["fold_trues"], best["fold_preds"],
            target_names=[remap_names.get(i, str(i)) for i in range(n_classes)],
            zero_division=0
        ))

        cm = confusion_matrix(best["fold_trues"], best["fold_preds"],
                              labels=list(range(n_classes)))
        print("Confusion Matrix (rows=true, cols=predicted):")
        header = f"{'':>8}" + "".join(
            f"{remap_names.get(i, str(i)):>9}" for i in range(n_classes))
        print(header)
        for i, row in enumerate(cm):
            label = remap_names.get(i, str(i))
            print(f"  {label:>6}  " + "".join(f"{v:>9}" for v in row))

    # ── SAVE JSON ─────────────────────────────────────────────────────────────
    safe_name = (task_name.replace(" ", "_")
                          .replace("(", "").replace(")", ""))
    save_path = Path(output_dir) / f"results_{safe_name}.json"

    def _jsonify(obj):
        if isinstance(obj, (np.integer,)):  return int(obj)
        if isinstance(obj, (np.floating,)): return float(obj)
        if isinstance(obj, np.ndarray):     return obj.tolist()
        return obj

    serialisable = {}
    for model_name, res in all_results.items():
        serialisable[model_name] = {
            k: (_jsonify(v)
                if isinstance(v, (np.ndarray, np.integer, np.floating))
                else [_jsonify(x) for x in v] if isinstance(v, list)
                else v)
            for k, v in res.items()
        }

    with open(save_path, "w") as f:
        json.dump(serialisable, f, indent=2)
    print(f"\nResults saved: {save_path}")

    return all_results

# ══════════════════════════════════════════════════════════════════════════════
# SECTION E — FEATURE IMPORTANCE
# ══════════════════════════════════════════════════════════════════════════════
def report_feature_importance(
    X: np.ndarray, y: np.ndarray,
    kept_names: np.ndarray, task_name: str, top_k: int = 20) -> None:
    print(f"\n{'═'*70}")
    print(f"FEATURE IMPORTANCE (RF, all-data fit) — {task_name}")
    print(f"  ⚠  For interpretation only. NOT used for evaluation.")
    print(f"{'─'*70}")

    scaler   = RobustScaler()
    X_scaled = scaler.fit_transform(X)

    rf = RandomForestClassifier(
        n_estimators=500, max_depth=12,
        class_weight="balanced",   
        n_jobs=-1, random_state=SEED
    )
    rf.fit(X_scaled, y)
    imp     = rf.feature_importances_
    top_idx = np.argsort(imp)[::-1][:top_k]

    print(f"  {'Rank':<5} {'Feature':<42} {'Importance':>12}")
    print(f"  {'─'*62}")
    for rank, i in enumerate(top_idx):
        print(f"  {rank+1:<5} {kept_names[i]:<42} {imp[i]:>12.4f}")

# ══════════════════════════════════════════════════════════════════════════════
# SECTION F — RUN ALL THREE TASKS
# ══════════════════════════════════════════════════════════════════════════════
# ── TASK A: Sleep Stage Classification (5-class) ──────────────────────────────
print("\n" + "█"*70)
print("TASK A: Sleep Stage Classification (5-class)")
print("█"*70)

results_stage = run_loso(
    X           = X_filtered,
    y           = y_stage,
    subject_ids = subject_ids,
    task_name   = "Sleep Stage Classification 5-class",
    label_names = STAGE_NAMES,
    output_dir  = OUTPUT_DIR,
)

report_feature_importance(
    X_filtered, y_stage, kept_names, "Sleep Stage Classification")

# ── TASK B: Insomnia Detection — all stages pooled (binary) ───────────────────
print("\n" + "█"*70)
print("TASK B: Insomnia Detection — All Stages Pooled (binary)")
print("█"*70)

results_insomnia = run_loso(
    X           = X_filtered,
    y           = y_group,
    subject_ids = subject_ids,
    task_name   = "Insomnia Detection Binary All Stages",
    label_names = GROUP_NAMES,
    output_dir  = OUTPUT_DIR,
)

report_feature_importance(
    X_filtered, y_group, kept_names, "Insomnia Detection")

# ── TASK C: Insomnia Detection — N2 only (stage-conditional hypothesis) ───────
print("\n" + "█"*70)
print("TASK C: Insomnia Detection — N2 Stage Only")
print("  Hypothesis: N2 spindle deficit is MORE discriminative than pooled signal")
print("█"*70)

n2_mask = y_stage == 2
print(f"  N2 epochs: {n2_mask.sum()}  "
      f"(Insomnia={((y_group==1)&n2_mask).sum()}, "
      f"Normal={((y_group==0)&n2_mask).sum()})")

results_n2 = run_loso(
    X           = X_filtered[n2_mask],
    y           = y_group[n2_mask],
    subject_ids = subject_ids[n2_mask],
    task_name   = "Insomnia Detection Binary N2 Only",
    label_names = GROUP_NAMES,
    output_dir  = OUTPUT_DIR,
)

# ══════════════════════════════════════════════════════════════════════════════
# SECTION G — CROSS-TASK SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*70}")
print("FINAL CROSS-TASK SUMMARY")
print(f"{'═'*70}")
print(f"{'Model':<20} {'StageAcc':>10} {'StageF1':>9} {'StageKappa':>12} "
      f"{'InsomniaF1':>12} {'N2F1':>8}")
print(f"{'─'*70}")

for mn in results_stage.keys():
    s  = results_stage.get(mn,    {})
    ig = results_insomnia.get(mn, {})
    n2 = results_n2.get(mn,       {})

    def _fmt(d, key): return f"{d[key]:.3f}" if key in d else "—"

    print(f"  {mn:<20}"
          f"  {_fmt(s,  'mean_acc'):>8}"
          f"  {_fmt(s,  'mean_f1'):>7}"
          f"  {_fmt(s,  'mean_kappa'):>10}"
          f"  {_fmt(ig, 'mean_f1'):>10}"
          f"  {_fmt(n2, 'mean_f1'):>6}")

print(f"\nAll result files saved to: {OUTPUT_DIR}/")
print(f"""
INTERPRETATION GUIDE:
  Kappa < 0.40          = poor — do not report as meaningful
  Kappa 0.40–0.60       = moderate — reportable with caveats
  Kappa > 0.60          = substantial — publication threshold
  p_vs_Dummy = 'ns'     = model is statistically indistinguishable from majority guesser
  Task C F1 > Task B F1 = stage-conditional insomnia hypothesis CONFIRMED (novel finding)
""")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# INSOMNIA DETECTION — CORRECTED EXPERIMENTAL DESIGNS (SHARMA 2021 FEATURES)
# Fixes the fundamental single-class test fold problem in Task B/C
# ═══════════════════════════════════════════════════════════════════════════════

import json, warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from scipy import stats as sp_stats
from scipy.stats import mannwhitneyu, wilcoxon
from sklearn.preprocessing    import RobustScaler
from sklearn.linear_model     import LogisticRegression
from sklearn.ensemble         import RandomForestClassifier
from sklearn.svm              import SVC
from sklearn.dummy            import DummyClassifier
from sklearn.base             import clone as sk_clone
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics          import (accuracy_score, cohen_kappa_score,
                                      f1_score, roc_auc_score,
                                      classification_report)
from sklearn.model_selection  import StratifiedKFold
from xgboost                  import XGBClassifier

warnings.filterwarnings('ignore')

# ── CONFIG ────────────────────────────────────────────────────────────────────
ML_DIR     = "/kaggle/working/ml_ready_sharma"
OUTPUT_DIR = "/kaggle/working/insomnia_corrected_sharma"
SEED       = 42
np.random.seed(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

STAGE_NAMES  = {0:"Wake", 1:"N1", 2:"N2", 3:"N3", 4:"REM"}
GROUP_NAMES  = {0:"Normal", 1:"Insomnia"}

# ── LOAD ──────────────────────────────────────────────────────────────────────
p           = Path(ML_DIR)
X           = np.load(p / "X.npy").astype(np.float64)
y_stage     = np.load(p / "y_stage.npy")
y_group     = np.load(p / "y_group.npy")
subject_ids = np.load(p / "subject_ids.npy")

with open(p / "metadata.json") as f:
    meta = json.load(f)

feature_names    = np.array(meta["feature_names"])
subject_registry = meta["subject_registry"]
id_to_name       = {v: k for k, v in subject_registry.items()}
n_subjects       = len(subject_registry)

# ── GLOBAL LOG1P TRANSFORM FOR WAVELET NORMS ──────────────────────────────────
# Sharma's amplitude norms are strictly positive but heavily right-skewed.
X = np.log1p(np.clip(X, 0, None))

from sklearn.feature_selection import VarianceThreshold
vt = VarianceThreshold(threshold=1e-6)
X  = vt.fit_transform(X).astype(np.float32)
kept_names = feature_names[vt.get_support()]
D = X.shape[1]

print(f"Features after drop: {D}  (102 expected for Sharma dataset)")
assert D == 24, f"Expected 102, got {D}"


# ══════════════════════════════════════════════════════════════════════════════
# DESIGN 1 — STATISTICAL FEATURE DISTRIBUTION TEST
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "█"*70)
print("DESIGN 1: Statistical Feature Distribution Test")
print("  Mann-Whitney U + Cohen's d per stage per feature")
print("  FDR correction: Benjamini-Hochberg (α=0.05)")
print("█"*70)

def cohens_d(a: np.ndarray, b: np.ndarray) -> float:
    """Pooled Cohen's d."""
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(
        ((na-1)*np.var(a,ddof=1) + (nb-1)*np.var(b,ddof=1)) / (na+nb-2+1e-12)
    )
    return float((a.mean() - b.mean()) / (pooled_std + 1e-12))


def fdr_bh(p_values: np.ndarray, alpha: float = 0.05) -> np.ndarray:
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = np.empty(n, dtype=int)
    ranked[order] = np.arange(1, n+1)
    threshold = (ranked / n) * alpha
    below = p_values <= threshold
    if not below.any():
        return np.zeros(n, dtype=bool)
    max_k = np.where(below)[0].max()
    significant = np.zeros(n, dtype=bool)
    significant[order[:max_k+1]] = True
    return significant


design1_results = {}

for stage_id, stage_name in STAGE_NAMES.items():
    mask_ins = (y_group == 1) & (y_stage == stage_id)
    mask_nor = (y_group == 0) & (y_stage == stage_id)

    n_ins = mask_ins.sum()
    n_nor = mask_nor.sum()

    if n_ins < 10 or n_nor < 10:
        print(f"\n  [{stage_name}] SKIP — insufficient epochs "
              f"(Insomnia={n_ins}, Normal={n_nor})")
        continue

    X_ins = X[mask_ins]
    X_nor = X[mask_nor]

    p_vals  = np.zeros(D)
    d_vals  = np.zeros(D)
    u_stats = np.zeros(D)

    for fi in range(D):
        u_stat, p = mannwhitneyu(X_ins[:, fi], X_nor[:, fi],
                                  alternative='two-sided')
        u_stats[fi] = u_stat
        p_vals[fi]  = p
        d_vals[fi]  = cohens_d(X_ins[:, fi], X_nor[:, fi])

    sig_mask = fdr_bh(p_vals, alpha=0.05)
    n_sig    = sig_mask.sum()

    sig_idx  = np.where(sig_mask)[0]
    if len(sig_idx) > 0:
        top_idx = sig_idx[np.argsort(np.abs(d_vals[sig_idx]))[::-1][:10]]
    else:
        top_idx = np.argsort(np.abs(d_vals))[::-1][:10]

    design1_results[stage_name] = {
        "n_insomnia": int(n_ins),
        "n_normal":   int(n_nor),
        "n_significant_features": int(n_sig),
        "top_features": [
            {"name": kept_names[i], "cohens_d": float(d_vals[i]),
             "p_raw": float(p_vals[i]), "significant_fdr": bool(sig_mask[i])}
            for i in top_idx
        ]
    }

    print(f"\n  [{stage_name}] Ins={n_ins} epochs | Nor={n_nor} epochs")
    print(f"  Significant features (FDR α=0.05): {n_sig}/{D}")
    print(f"  {'Feature':<40} {'Cohen d':>9} {'p_raw':>12} {'FDR sig':>8}")
    print(f"  {'─'*72}")
    for i in top_idx:
        sig_str = "YES" if sig_mask[i] else "no"
        d_str   = f"{d_vals[i]:>+9.3f}"
        p_str   = f"{p_vals[i]:>12.2e}"
        print(f"  {kept_names[i]:<40} {d_str} {p_str} {sig_str:>8}")

with open(f"{OUTPUT_DIR}/design1_statistical_tests.json", "w") as f:
    json.dump(design1_results, f, indent=2)
print(f"\nDesign 1 results saved to {OUTPUT_DIR}/design1_statistical_tests.json")


# ══════════════════════════════════════════════════════════════════════════════
# DESIGN 2 — SUBJECT-LEVEL FEATURE AGGREGATION + LOSO
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "█"*70)
print("DESIGN 2: Subject-Level Aggregation + LOSO (N=22)")
print(f"  Each subject → mean+std feature vector ({D * 2} dims)")
print("  LOSO: 22-fold, train=21 subjects, test=1 subject")
print("█"*70)

X_subj  = np.zeros((n_subjects, D * 2), dtype=np.float32)
y_subj  = np.zeros(n_subjects,          dtype=np.int32)
names_subj = (
    [f"mean_{n}" for n in kept_names] +
    [f"std_{n}"  for n in kept_names]
)

for sid in range(n_subjects):
    mask          = subject_ids == sid
    epochs        = X[mask]
    X_subj[sid]   = np.concatenate([epochs.mean(axis=0),
                                     epochs.std(axis=0)])
    y_subj[sid]   = int(y_group[mask][0])  

print(f"\n  Subject matrix: {X_subj.shape}  "
      f"(Normal={( y_subj==0).sum()}, Insomnia={(y_subj==1).sum()})")

assert np.isnan(X_subj).sum() == 0
assert np.isinf(X_subj).sum() == 0

def get_subj_models(seed: int = SEED) -> dict:
    return {
        "Dummy":    DummyClassifier(strategy="most_frequent", random_state=seed),
        "LogReg":   LogisticRegression(C=0.1, max_iter=2000, solver="lbfgs",
                                        class_weight="balanced", random_state=seed),
        "SVM_RBF":  SVC(C=1.0, kernel="rbf", gamma="scale", probability=True,
                         class_weight="balanced", random_state=seed),
        "RF":       RandomForestClassifier(n_estimators=200, max_depth=4,
                                            class_weight="balanced",
                                            n_jobs=-1, random_state=seed),
        "XGBoost":  XGBClassifier(n_estimators=100, max_depth=3,
                                   learning_rate=0.1, eval_metric="logloss",
                                   n_jobs=-1, random_state=seed, verbosity=0,
                                   objective="binary:logistic"),
    }

results_subj = {name: {"preds":[], "trues":[], "probs":[],
                        "accs":[], "subj_names":[]}
                for name in get_subj_models()}

print(f"\n  {'Subject':<20} {'Group':>9}  " +
      "  ".join(f"{m:>8}" for m in get_subj_models()))
print("  " + "─"*75)

for test_sid in range(n_subjects):
    train_mask = np.arange(n_subjects) != test_sid
    test_mask  = np.array([test_sid])

    X_tr = X_subj[train_mask]
    X_te = X_subj[test_mask]
    y_tr = y_subj[train_mask]
    y_te = y_subj[test_mask]

    if len(np.unique(y_tr)) < 2:
        continue

    scaler = RobustScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    subj_name = id_to_name[test_sid]
    group_str = GROUP_NAMES[int(y_te[0])]
    fold_line = f"  {subj_name:<20} {group_str:>9}  "

    for model_name, model in get_subj_models().items():
        clf = sk_clone(model)
        if "XGBoost" in model_name:
            sw = compute_sample_weight('balanced', y_tr)
            clf.fit(X_tr_s, y_tr, sample_weight=sw)
        else:
            clf.fit(X_tr_s, y_tr)

        pred = clf.predict(X_te_s)[0]
        correct = int(pred == y_te[0])

        try:
            prob = clf.predict_proba(X_te_s)[0][1]
        except Exception:
            prob = float(pred)

        results_subj[model_name]["preds"].append(int(pred))
        results_subj[model_name]["trues"].append(int(y_te[0]))
        results_subj[model_name]["probs"].append(float(prob))
        results_subj[model_name]["accs"].append(correct)
        results_subj[model_name]["subj_names"].append(subj_name)

        fold_line += f"  {'✓' if correct else '✗':>8}"

    print(fold_line)

print(f"\n  {'─'*75}")
print(f"\n  DESIGN 2 SUMMARY (N=22 LOSO, subject-level classification)")
print(f"  {'Model':<12} {'Acc':>6} {'AUC':>6} {'F1':>6} {'Correct/22':>12}")
print(f"  {'─'*50}")

for model_name, res in results_subj.items():
    trues  = np.array(res["trues"])
    preds  = np.array(res["preds"])
    probs  = np.array(res["probs"])
    accs   = np.array(res["accs"])

    acc  = accs.mean()
    f1   = f1_score(trues, preds, average="macro", zero_division=0)
    n_correct = accs.sum()

    try:
        auc = roc_auc_score(trues, probs)
    except Exception:
        auc = float('nan')

    res["mean_acc"] = float(acc)
    res["auc"]      = float(auc)
    res["mean_f1"]  = float(f1)
    res["n_correct"]= int(n_correct)

    print(f"  {model_name:<12} {acc:>6.3f} {auc:>6.3f} {f1:>6.3f} "
          f"{n_correct:>6}/22")

print(f"\n  Binomial significance test (H0: accuracy = 0.5, one-tailed)")
print(f"  {'Model':<12} {'Correct':>8} {'p_binom':>10} {'Sig':>6}")
print(f"  {'─'*40}")
from scipy.stats import binomtest
for model_name, res in results_subj.items():
    if model_name == "Dummy":
        continue
    n_c = res["n_correct"]
    bt  = binomtest(n_c, n=22, p=0.5, alternative='greater')
    sig = ("*" if bt.pvalue < 0.05 else "ns")
    print(f"  {model_name:<12} {n_c:>8}/22  {bt.pvalue:>10.4f} {sig:>6}")

with open(f"{OUTPUT_DIR}/design2_subject_loso.json", "w") as f:
    json.dump({k: {kk: (vv if not isinstance(vv, np.ndarray) else vv.tolist())
                   for kk, vv in v.items()}
               for k, v in results_subj.items()}, f, indent=2)
print(f"\nDesign 2 results saved.")


# ══════════════════════════════════════════════════════════════════════════════
# DESIGN 3 — STAGE-CONDITIONAL EPOCH-LEVEL CV 
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "█"*70)
print("DESIGN 3: Stage-Conditional Epoch CV (within-stage, 5-fold stratified)")
print("  ⚠ Subject identity leaks into train/test — upper bound only")
print("█"*70)

design3_results = {}

for stage_id, stage_name in STAGE_NAMES.items():
    stage_mask = y_stage == stage_id
    X_s   = X[stage_mask]
    y_s   = y_group[stage_mask]      
    sid_s = subject_ids[stage_mask]

    n_ins = (y_s == 1).sum()
    n_nor = (y_s == 0).sum()

    if n_ins < 20 or n_nor < 20:
        continue

    print(f"\n  [{stage_name}] Total={len(y_s)}  Ins={n_ins}  Nor={n_nor}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_results = defaultdict(list)

    for fold_i, (tr_idx, te_idx) in enumerate(skf.split(X_s, y_s)):
        X_tr, X_te = X_s[tr_idx], X_s[te_idx]
        y_tr, y_te = y_s[tr_idx], y_s[te_idx]

        scaler = RobustScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_te_s = scaler.transform(X_te)

        sw = compute_sample_weight('balanced', y_tr)

        for model_name, model in [
            ("LogReg",  LogisticRegression(C=1.0, class_weight="balanced",
                                            max_iter=1000, random_state=SEED)),
            ("RF",      RandomForestClassifier(n_estimators=200, max_depth=8,
                                               class_weight="balanced",
                                               n_jobs=-1, random_state=SEED)),
            ("XGBoost", XGBClassifier(n_estimators=200, max_depth=5,
                                       learning_rate=0.05, eval_metric="logloss",
                                       n_jobs=-1, random_state=SEED,
                                       verbosity=0, objective="binary:logistic")),
        ]:
            clf = sk_clone(model)
            if "XGBoost" in model_name:
                clf.fit(X_tr_s, y_tr, sample_weight=sw)
            else:
                clf.fit(X_tr_s, y_tr)

            y_pred = clf.predict(X_te_s)
            acc    = accuracy_score(y_te, y_pred)
            kappa  = cohen_kappa_score(y_te, y_pred)
            f1     = f1_score(y_te, y_pred, average="macro", zero_division=0)

            try:
                auc = roc_auc_score(y_te, clf.predict_proba(X_te_s)[:, 1])
            except Exception:
                auc = float('nan')

            fold_results[model_name].append(
                {"acc": acc, "kappa": kappa, "f1": f1, "auc": auc}
            )

    print(f"  {'Model':<10} {'Acc±SD':>12} {'Kappa±SD':>12} "
          f"{'F1±SD':>12} {'AUC±SD':>12}")
    print(f"  {'─'*55}")
    stage_res = {}
    for model_name, folds in fold_results.items():
        accs   = np.array([f["acc"]   for f in folds])
        kappas = np.array([f["kappa"] for f in folds])
        f1s    = np.array([f["f1"]    for f in folds])
        aucs   = np.array([f["auc"]   for f in folds])

        print(f"  {model_name:<10} "
              f"{accs.mean():.3f}±{accs.std():.3f}  "
              f"{kappas.mean():.3f}±{kappas.std():.3f}  "
              f"{f1s.mean():.3f}±{f1s.std():.3f}  "
              f"{aucs.mean():.3f}±{aucs.std():.3f}")

        stage_res[model_name] = {
            "mean_acc":   float(accs.mean()),
            "mean_kappa": float(kappas.mean()),
            "mean_f1":    float(f1s.mean()),
            "mean_auc":   float(aucs.mean()),
        }

    design3_results[stage_name] = stage_res

with open(f"{OUTPUT_DIR}/design3_stage_cv.json", "w") as f:
    json.dump(design3_results, f, indent=2)


# ══════════════════════════════════════════════════════════════════════════════
# FINAL SYNTHESIS
# ══════════════════════════════════════════════════════════════════════════════
print(f"\n{'═'*70}")
print("SYNTHESIS: Which stages discriminate Insomnia vs Normal?")
print(f"{'═'*70}")
print(f"  {'Stage':<8} {'Sig Features':>13} {'D3 XGB Kappa':>14} {'D3 XGB AUC':>12}")
print(f"  {'─'*50}")
for stage_name in STAGE_NAMES.values():
    d1 = design1_results.get(stage_name, {})
    d3 = design3_results.get(stage_name, {}).get("XGBoost", {})
    n_sig   = d1.get("n_significant_features", "—")
    d3k     = f"{d3.get('mean_kappa', float('nan')):.3f}" if d3 else "—"
    d3auc   = f"{d3.get('mean_auc',   float('nan')):.3f}" if d3 else "—"
    print(f"  {stage_name:<8} {str(n_sig):>13} {d3k:>14} {d3auc:>12}")

print(f"""
KEY: 
  Sig Features  = features with FDR-corrected p<0.05 (Design 1)
  D3 XGB Kappa  = 5-fold CV Kappa within stage epochs (Design 2 — upper bound)
  D3 XGB AUC    = AUC (0.5=chance, 1.0=perfect)

INTERPRETATION RULES:
  AUC > 0.70 AND Kappa > 0.30 → stage has discriminative signal
  AUC < 0.60                  → no discriminative signal in this stage

USE Design 2 (subject-level LOSO) as your PRIMARY result.
USE Design 1 (Mann-Whitney) as your biomarker discovery result.
DO NOT report Design 3 as LOSO — it is an upper bound only.

All results saved to: {OUTPUT_DIR}/
""")